In [0]:
df = spark.read.csv(
    "/Volumes/mlops_course/ingesting_data/files/DailyDelhiClimateTrain.csv",
    header=True,
    inferSchema=True
)

df_sorted = df.orderBy("date")



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Adding row number for making the batches
w = Window.orderBy("date")
df_numbered = df_sorted.withColumn("rn", row_number().over(w))

total_rows = df_numbered.count()
batches = 5
batch_size = total_rows / batches

# Creating the batches
for i in range(batches):
    start = i * batch_size + 1
    end = (i + 1) * batch_size if i < 4 else total_rows

    batch_df = df_numbered.filter(
        (df_numbered.rn >= start) & (df_numbered.rn <= end)
    )

    batch_df.write.mode("overwrite").option("header", True).csv(
        f"/Volumes/mlops_course/ingesting_data/staging_batch_files/batch_{i+1}"
    )


